# AHR v2 · Scout and Worker neural foundation

This fresh, Go-specific network shares a residual convolutional encoder. The Scout assigns energies to every board move and pass; the Worker assigns energies to loss, draw and win. Search uses **win probability minus loss probability**. Score and ownership remain auxiliary predictions.

**Inputs:** exported definitions from `00_contracts_and_recovery.ipynb` and `01_rules_and_replay.ipynb`, PyTorch and NumPy. **Outputs:** native fresh-weight evaluation APIs and a CPU validation receipt in `reports/ahr_v2/network_validation.json`. **Expected runtime:** under two minutes on CPU. No learned weights from earlier experiments are imported.

**Checkpoint and resume:** this notebook has no durable training stage or model checkpoint. Its three optimizer steps use disposable legal fixtures only. Run All in a fresh `ahr-go` kernel to repeat the checks. Durable future training must use notebook 00's atomic, single-writer checkpoint contract; all state, optimizer, scheduler, scaler, random states and data position must resume together.

The initial rules are area scoring, no suicide and positional superko. The main board is 9×9 with komi 7.5. Native spatial shapes also support smaller validation boards. Different board sizes are grouped into separate forward batches; no padded intersections enter pooling. Truncated games have no terminal targets.

## Load definitions only

The import loader executes only cells tagged `ahr-export`. It does not run upstream fixtures, training or generation.

In [ ]:
from pathlib import Path
import nbformat

PROJECT = Path.cwd()
while not (PROJECT / 'experiments' / 'ahr_v2').is_dir():
    if PROJECT.parent == PROJECT:
        raise FileNotFoundError('Run this notebook inside the AHR Go workspace.')
    PROJECT = PROJECT.parent

contracts = PROJECT / 'experiments/ahr_v2/00_contracts_and_recovery.ipynb'
for cell in nbformat.read(contracts, as_version=4).cells:
    if cell.cell_type == 'code' and 'ahr-export' in cell.metadata.get('tags', []):
        exec(compile(cell.source, str(contracts), 'exec'), globals())
_ = go2_import_exports(PROJECT / 'experiments/ahr_v2/01_rules_and_replay.ipynb', globals())


## Encode board and rule context

All stone, history, liberty, score and ownership channels use the **current player** as their perspective. Earlier boards are not reinterpreted using the player who moved then. Legality is supplied by the exact engine, including its complete positional history. The complete history is also part of the neural cache key, even though only four previous boards are visible to the encoder.

In [ ]:
from dataclasses import dataclass, asdict
import hashlib
import json
import math
import time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

GO2_FEATURE_NAMES = (
    'own_stones', 'opponent_stones',
    'own_previous_1', 'opponent_previous_1',
    'own_previous_2', 'opponent_previous_2',
    'own_previous_3', 'opponent_previous_3',
    'own_previous_4', 'opponent_previous_4',
    'own_liberty_1', 'own_liberty_2', 'own_liberty_3plus',
    'opponent_liberty_1', 'opponent_liberty_2', 'opponent_liberty_3plus',
    'legal_board_moves', 'komi_for_current_player', 'black_to_play',
    'area_no_suicide_psk', 'consecutive_pass_fraction',
)
GO2_FEATURE_COUNT = len(GO2_FEATURE_NAMES)
GO2_OUTCOMES = ('loss', 'draw', 'win')


def go2_network_terminal(state):
    return int(state.consecutive_passes) >= 2


def go2_liberty_planes(board, size, player):
    board = tuple(int(x) for x in board)
    planes = np.zeros((3, size * size), dtype=np.float32)
    visited = set()
    for start, color in enumerate(board):
        if color != player or start in visited:
            continue
        group, liberties, todo = set(), set(), [start]
        while todo:
            point = todo.pop()
            if point in group:
                continue
            group.add(point)
            row, col = divmod(point, size)
            neighbors = []
            if row: neighbors.append(point-size)
            if row+1 < size: neighbors.append(point+size)
            if col: neighbors.append(point-1)
            if col+1 < size: neighbors.append(point+1)
            for neighbor in neighbors:
                if board[neighbor] == 0:
                    liberties.add(neighbor)
                elif board[neighbor] == player and neighbor not in group:
                    todo.append(neighbor)
        visited.update(group)
        if not liberties:
            raise ValueError('An occupied group without liberties is not a legal Go position.')
        planes[min(len(liberties), 3)-1, list(group)] = 1.0
    return planes.reshape(3, size, size)


def go2_encode_state(state):
    if state.rule_profile != 'area_no_suicide_psk_v1':
        raise ValueError('This feature schema supports only area_no_suicide_psk_v1.')
    size, player = int(state.size), int(state.to_play)
    if size < 2 or player not in (-1, 1):
        raise ValueError('Invalid board size or player.')
    board = np.asarray(state.board, dtype=np.int8).reshape(size, size)
    if not np.isin(board, (-1, 0, 1)).all():
        raise ValueError('Invalid stone color.')
    features = np.zeros((GO2_FEATURE_COUNT, size, size), dtype=np.float32)
    features[0], features[1] = board == player, board == -player
    recent = list(state.recent_boards)
    if recent and tuple(recent[-1]) != tuple(state.board):
        raise ValueError('Recent board history must end with the current board.')
    previous = recent[:-1] if recent else []
    for offset in range(4):
        if len(previous) > offset:
            historic = np.asarray(previous[-1-offset], dtype=np.int8).reshape(size, size)
            features[2+2*offset], features[3+2*offset] = historic == player, historic == -player
    features[10:13] = go2_liberty_planes(state.board, size, player)
    features[13:16] = go2_liberty_planes(state.board, size, -player)
    legal = np.asarray(go2_legal_mask(state), dtype=bool)
    if legal.shape != (size*size+1,):
        raise ValueError('Legal mask does not match the board.')
    features[16] = legal[:-1].reshape(size, size)
    features[17].fill(-player * float(state.komi) / (size*size + abs(float(state.komi))))
    features[18].fill(float(player == 1))
    features[19].fill(1.0)
    features[20].fill(min(int(state.consecutive_passes), 2)/2.0)
    if not np.isfinite(features).all():
        raise ValueError('Non-finite feature.')
    return features


## Shared encoder and energy heads

Lower Scout energy increases an action's prior. Lower Worker energy makes that outcome more probable, so a low loss energy represents a bad position. Every legal move remains eligible. No tactical heuristic overwrites either neural energy head.

In [ ]:
@dataclass(frozen=True)
class Go2NetConfig:
    board_size: int = 9
    width: int = 64
    blocks: int = 4
    scout_temperature: float = 1.0
    worker_temperature: float = 1.0
    rule_profile: str = 'area_no_suicide_psk_v1'

    def __post_init__(self):
        if self.board_size < 2 or self.width < 8 or self.blocks < 1:
            raise ValueError('Board size >=2, width >=8 and at least one residual block are required.')
        if not math.isfinite(self.scout_temperature) or self.scout_temperature <= 0:
            raise ValueError('Scout temperature must be positive and finite.')
        if not math.isfinite(self.worker_temperature) or self.worker_temperature <= 0:
            raise ValueError('Worker temperature must be positive and finite.')
        if self.rule_profile != 'area_no_suicide_psk_v1':
            raise ValueError('Unsupported rule profile.')


def go2_group_count(width):
    return next(groups for groups in (8, 4, 2, 1) if width % groups == 0)


class Go2ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        groups = go2_group_count(width)
        self.conv1 = nn.Conv2d(width, width, 3, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(groups, width)
        self.conv2 = nn.Conv2d(width, width, 3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(groups, width)

    def forward(self, features):
        residual = F.silu(self.norm1(self.conv1(features)))
        residual = self.norm2(self.conv2(residual))
        return F.silu(features + residual)


class Go2Network(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = config or Go2NetConfig()
        width = self.config.width
        self.stem = nn.Sequential(
            nn.Conv2d(GO2_FEATURE_COUNT, width, 3, padding=1, bias=False),
            nn.GroupNorm(go2_group_count(width), width), nn.SiLU(),
        )
        self.encoder = nn.Sequential(*[Go2ResidualBlock(width) for _ in range(self.config.blocks)])
        self.scout_board = nn.Conv2d(width, 1, 1)
        self.scout_pass = nn.Linear(2*width, 1)
        self.worker = nn.Sequential(nn.Linear(2*width, width), nn.SiLU(), nn.Linear(width, 3))
        self.score_head = nn.Sequential(nn.Linear(2*width, width), nn.SiLU(), nn.Linear(width, 1))
        self.ownership_head = nn.Conv2d(width, 1, 1)
        self._go2_identity_stamp = None
        self._go2_identity_digest = None

    def forward(self, features):
        if features.ndim != 4 or features.shape[1] != GO2_FEATURE_COUNT:
            raise ValueError('Features must have shape [batch,21,size,size].')
        if features.shape[-1] != features.shape[-2] or features.shape[-1] < 2:
            raise ValueError('Each forward batch must have one native square board size.')
        encoded = self.encoder(self.stem(features))
        global_features = torch.cat((encoded.mean(dim=(2,3)), encoded.amax(dim=(2,3))), dim=1)
        scout_energy = torch.cat((self.scout_board(encoded).flatten(1), self.scout_pass(global_features)), dim=1)
        return {
            'scout_energy': scout_energy,
            'worker_energy': self.worker(global_features),
            'score': torch.tanh(self.score_head(global_features)).squeeze(1),
            'ownership': torch.tanh(self.ownership_head(encoded)).flatten(1),
            'global_features': global_features,
        }


def go2_model_identity(model, force=False):
    # Optimizer updates and load_state_dict increment tensor versions. Hash all
    # tensor bytes once per frozen version, rather than at every searched leaf.
    tensors = tuple(model.parameters()) + tuple(model.buffers())
    config_json = json.dumps(asdict(model.config), sort_keys=True, separators=(',', ':'))
    stamp = (config_json, tuple((id(tensor), tensor._version, str(tensor.device), str(tensor.dtype)) for tensor in tensors))
    if force or model._go2_identity_stamp != stamp:
        digest = hashlib.sha256(config_json.encode())
        for name, tensor in sorted(model.state_dict().items()):
            raw = tensor.detach().cpu().contiguous()
            digest.update(name.encode())
            digest.update(str(raw.dtype).encode())
            digest.update(json.dumps(list(raw.shape)).encode())
            digest.update(raw.reshape(-1).view(torch.uint8).numpy().tobytes())
        model._go2_identity_digest = digest.hexdigest()
        model._go2_identity_stamp = stamp
    return model._go2_identity_digest


def go2_invalidate_model_identity(model):
    # Required after unsupported direct .data replacement or external weight edits.
    model._go2_identity_stamp = None


## Batched evaluation for search

Terminal states are scored by the exact engine and are rejected here. The cache reference combines the full rules/history identity with a hash of the actual network weights. Auxiliary score is normalized by board area plus absolute komi; ownership lies in [-1,1]. Neither changes the Worker value used by search.

In [ ]:
def go2_eval_batch(model, states):
    states = list(states)
    if not states:
        return []
    for state in states:
        if go2_network_terminal(state):
            raise ValueError('Terminal states must use exact outcome scoring, never a neural leaf evaluation.')
        if state.rule_profile != model.config.rule_profile:
            raise ValueError('Model and position rule profiles differ.')
    version = go2_model_identity(model)
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    groups = {}
    for index, state in enumerate(states):
        groups.setdefault(int(state.size), []).append((index, state))
    results = [None] * len(states)
    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            for size, members in groups.items():
                features = torch.as_tensor(np.stack([go2_encode_state(s) for _, s in members]), device=device, dtype=dtype)
                legal = torch.as_tensor(np.stack([go2_legal_mask(s) for _, s in members]), device=device, dtype=torch.bool)
                if not legal.any(dim=1).all():
                    raise ValueError('A nonterminal state must have a legal action.')
                raw = model(features)
                if not all(torch.isfinite(value).all().item() for value in raw.values()):
                    raise ValueError('Non-finite neural output.')
                scout_logits = (-raw['scout_energy']/model.config.scout_temperature).masked_fill(~legal, -torch.inf)
                prior = torch.softmax(scout_logits, dim=1).masked_fill(~legal, 0.0)
                outcome = torch.softmax(-raw['worker_energy']/model.config.worker_temperature, dim=1)
                value = outcome[:,2] - outcome[:,0]
                arrays = {name: tensor.detach().cpu().float().numpy() for name, tensor in (
                    ('prior', prior), ('outcome', outcome), ('value', value), ('score', raw['score']),
                    ('ownership', raw['ownership']), ('features', raw['global_features']),
                )}
                masks = legal.cpu().numpy()
                for batch_index, (original_index, state) in enumerate(members):
                    results[original_index] = {
                        'prior': arrays['prior'][batch_index].copy(),
                        'value': float(arrays['value'][batch_index]),
                        'outcome_probabilities': arrays['outcome'][batch_index].copy(),
                        'score': float(arrays['score'][batch_index]),
                        'ownership': arrays['ownership'][batch_index].copy(),
                        'global_features': arrays['features'][batch_index].copy(),
                        'legal_mask': masks[batch_index].copy(),
                        'cache_key': go2_cache_key(state),
                        'network_version': version,
                        'value_perspective': 'current_player',
                        'score_normalization': float(size*size+abs(float(state.komi))),
                        'score_perspective': 'current_player',
                        'ownership_perspective': 'current_player',
                    }
    finally:
        model.train(was_training)
    return results


go2_model_evaluate = go2_eval_batch


## Train from typed, traceable labels

A policy label states whether it is a bounded verified target, a search estimate or a teacher estimate. Outcome, score and ownership identify either a replay-verified terminal result or a teacher estimate. Teacher estimates require the pinned teacher hash and search budget. Missing labels are omitted from their loss denominator; an UNKNOWN game is never converted into a loss or draw.

The terminal helper replays the remaining actions from the sampled state and checks the terminal identity before deriving current-player signs. The joint loss accepts distributions for both energy heads, smooth-L1 score loss and ownership mean-square loss.

The loss consumes labels that were validated during ingestion. Provenance fields identify their source and certainty; their presence alone does not independently prove a target. Exact terminal labels should be built with the replay helper. Any teacher or search target remains an estimate.

In [ ]:
@dataclass(frozen=True)
class Go2LossConfig:
    policy_weight: float = 1.0
    outcome_weight: float = 1.0
    score_weight: float = 0.25
    ownership_weight: float = 0.25
    l2_weight: float = 0.0

    def __post_init__(self):
        if any(not math.isfinite(value) or value < 0 for value in asdict(self).values()):
            raise ValueError('Loss weights must be finite and nonnegative.')
        if sum(asdict(self).values()) <= 0:
            raise ValueError('At least one loss weight must be positive.')


def go2_label_provenance(label, kind):
    provenance = dict(label.get('provenance', {}))
    if not all(isinstance(provenance.get(key), str) and provenance[key].strip() for key in ('source', 'source_id')):
        raise ValueError('Every label needs a nonempty source and source_id.')
    required_perspective = 'action' if kind == 'policy' else 'current_player'
    if provenance.get('perspective') != required_perspective:
        raise ValueError('Label perspective is inconsistent with its head.')
    certainty = provenance.get('certainty')
    allowed = {'exact_bounded', 'search_estimate', 'teacher_estimate'} if kind == 'policy' else {'terminal_verified', 'teacher_estimate'}
    if certainty not in allowed:
        raise ValueError('Unsupported label certainty; UNKNOWN has no supervised label.')
    if certainty == 'exact_bounded' and not provenance.get('scope'):
        raise ValueError('A bounded policy label needs its exact bounded scope.')
    if certainty == 'teacher_estimate':
        if not provenance.get('teacher_sha256') or not isinstance(provenance.get('search_budget'), dict) or not provenance['search_budget']:
            raise ValueError('Teacher labels need a pinned hash and explicit search budget.')
    return provenance


def go2_training_row(state, terminal_state=None, policy_target=None, policy_provenance=None, source_id=None):
    if go2_network_terminal(state):
        raise ValueError('Search/training positions must be nonterminal.')
    go2_verify_replay(state)
    labels = {}
    if policy_target is not None:
        labels['policy'] = {'target': [float(value) for value in policy_target], 'provenance': dict(policy_provenance or {})}
        go2_label_provenance(labels['policy'], 'policy')
    if terminal_state is not None:
        if not go2_network_terminal(terminal_state):
            raise ValueError('A cutoff or UNKNOWN game cannot supply terminal labels.')
        if state.size != terminal_state.size or state.komi != terminal_state.komi or state.rule_profile != terminal_state.rule_profile:
            raise ValueError('Sample and terminal rules differ.')
        if state.initial_board != terminal_state.initial_board or state.initial_to_play != terminal_state.initial_to_play:
            raise ValueError('Sample and terminal initial positions differ.')
        if tuple(terminal_state.actions[:len(state.actions)]) != tuple(state.actions):
            raise ValueError('Sample is not an action prefix of this terminal game.')
        replay = state
        for action in terminal_state.actions[len(state.actions):]:
            replay = go2_step(replay, int(action))
        if go2_cache_key(replay) != go2_cache_key(terminal_state):
            raise ValueError('Terminal replay identity mismatch.')
        outcome = go2_outcome(terminal_state)
        score = go2_score(terminal_state)
        current_value = int(state.to_play) * int(outcome['value_black'])
        outcome_target = [float(current_value == -1), float(current_value == 0), float(current_value == 1)]
        provenance = {
            'source': 'exact_terminal_replay',
            'source_id': str(source_id or go2_state_digest(terminal_state)),
            'perspective': 'current_player', 'certainty': 'terminal_verified',
            'terminal_state_sha256': go2_state_digest(terminal_state),
            'position_state_sha256': go2_state_digest(state),
        }
        labels['outcome'] = {'target': outcome_target, 'provenance': dict(provenance)}
        labels['score'] = {'target': int(state.to_play) * float(score['score_black']) / (state.size*state.size+abs(float(state.komi))), 'provenance': dict(provenance)}
        labels['ownership'] = {'target': (int(state.to_play)*np.asarray(score['ownership_black'], dtype=np.float32)).tolist(), 'provenance': dict(provenance)}
    if not labels:
        raise ValueError('At least one typed label is required.')
    return {'state': go2_state_to_dict(state), 'labels': labels}


def go2_probability_target(target, length, legal=None):
    array = np.asarray(target, dtype=np.float32)
    if array.shape != (length,) or not np.isfinite(array).all() or (array < 0).any():
        raise ValueError('A probability target must be finite, nonnegative and correctly shaped.')
    if not np.isclose(float(array.sum()), 1.0, atol=1e-5, rtol=0):
        raise ValueError('A probability target must sum to one.')
    if legal is not None and np.any(array[~np.asarray(legal, dtype=bool)] != 0):
        raise ValueError('An illegal action has nonzero policy target mass.')
    return array


def go2_joint_loss(model, rows, config=None):
    config = config or Go2LossConfig()
    rows = list(rows)
    if not rows:
        raise ValueError('A training batch must contain rows.')
    device, dtype = next(model.parameters()).device, next(model.parameters()).dtype
    groups = {}
    for row in rows:
        state = go2_state_from_dict(row['state']) if isinstance(row['state'], dict) else row['state']
        if go2_network_terminal(state):
            raise ValueError('Terminal positions use exact scoring and must not train a searched neural leaf.')
        labels = row.get('labels', {})
        if not labels or set(labels) - {'policy', 'outcome', 'score', 'ownership'}:
            raise ValueError('Unknown or empty label schema.')
        for kind, label in labels.items():
            go2_label_provenance(label, kind)
        groups.setdefault(state.size, []).append((state, labels))
    terms = {kind: [] for kind in ('policy', 'outcome', 'score', 'ownership')}
    for size, members in groups.items():
        features = torch.as_tensor(np.stack([go2_encode_state(state) for state, _ in members]), device=device, dtype=dtype)
        legal = torch.as_tensor(np.stack([go2_legal_mask(state) for state, _ in members]), device=device, dtype=torch.bool)
        raw = model(features)
        if not all(torch.isfinite(tensor).all().item() for tensor in raw.values()):
            raise ValueError('Non-finite training output.')
        scout_logits = (-raw['scout_energy']/model.config.scout_temperature).masked_fill(~legal, -torch.inf)
        log_policy = F.log_softmax(scout_logits, dim=1).masked_fill(~legal, 0.0)
        log_outcome = F.log_softmax(-raw['worker_energy']/model.config.worker_temperature, dim=1)
        for index, (state, labels) in enumerate(members):
            if 'policy' in labels:
                target = go2_probability_target(labels['policy']['target'], size*size+1, legal[index].detach().cpu().numpy())
                target = torch.as_tensor(target, device=device, dtype=dtype)
                terms['policy'].append(-(target * log_policy[index]).sum())
            if 'outcome' in labels:
                target = torch.as_tensor(go2_probability_target(labels['outcome']['target'], 3), device=device, dtype=dtype)
                terms['outcome'].append(-(target * log_outcome[index]).sum())
            if 'score' in labels:
                target_value = float(labels['score']['target'])
                if not math.isfinite(target_value) or abs(target_value) > 1+1e-6:
                    raise ValueError('Normalized score target must lie in [-1,1].')
                target = torch.as_tensor(target_value, device=device, dtype=dtype)
                terms['score'].append(F.smooth_l1_loss(raw['score'][index], target))
            if 'ownership' in labels:
                target_array = np.asarray(labels['ownership']['target'], dtype=np.float32)
                if target_array.shape != (size*size,) or not np.isfinite(target_array).all() or np.abs(target_array).max() > 1+1e-6:
                    raise ValueError('Ownership target must contain one current-player value in [-1,1] per point.')
                target = torch.as_tensor(target_array, device=device, dtype=dtype)
                terms['ownership'].append(F.mse_loss(raw['ownership'][index], target))
    zero = next(model.parameters()).sum() * 0.0
    components = {kind: torch.stack(values).mean() if values else zero for kind, values in terms.items()}
    regularization = torch.stack([parameter.square().mean() for parameter in model.parameters()]).mean() if config.l2_weight else zero
    total = (config.policy_weight*components['policy'] + config.outcome_weight*components['outcome']
             + config.score_weight*components['score'] + config.ownership_weight*components['ownership']
             + config.l2_weight*regularization)
    if not torch.isfinite(total):
        raise ValueError('Non-finite joint loss.')
    return {'loss': total, 'components': components, 'regularization': regularization,
            'counts': {kind: len(values) for kind, values in terms.items()}}


## CPU integration checks

These fixtures check player signs, exact terminal replay, complete legal masking, history-sensitive references, native batch sizes and gradient flow through all four heads. Three disposable optimizer steps test that this joint objective can decrease; they do not establish playing strength.

In [ ]:
def go2_validate_network(project=None):
    project = go2_project(project)
    start_time = time.perf_counter()
    prior_threads = torch.get_num_threads()
    torch.set_num_threads(min(2, prior_threads))
    try:
        go2_seed(2026100602)
        model = Go2Network(Go2NetConfig(width=16, blocks=2)).cpu()
        initial = go2_initial_state(size=9, komi=7.5)
        passed = go2_step(initial, 81)
        empty_terminal = go2_step(passed, 81)
        occupied = go2_step(initial, 40)
        occupied_terminal = go2_step(go2_step(occupied, 81), 81)
        small = go2_initial_state(size=5, komi=0.5)
        predictions = go2_eval_batch(model, [initial, occupied, small, passed])
        checks = []
        assert [len(item['prior']) for item in predictions] == [82,82,26,82]
        assert all(np.isclose(item['prior'].sum(), 1.0, atol=1e-6) for item in predictions)
        assert all(np.all(item['prior'][~item['legal_mask']] == 0) for item in predictions)
        assert predictions[1]['prior'][40] == 0
        assert all(np.isclose(item['value'], item['outcome_probabilities'][2]-item['outcome_probabilities'][0], atol=1e-7) for item in predictions)
        checks.append('Native mixed-size grouping, strict zero illegal priors and Worker W-L value.')
        features = go2_encode_state(occupied)
        assert features.shape == (21,9,9)
        assert features[1,4,4] == 1 and features[0,4,4] == 0
        next_state = go2_step(occupied, 0)
        history = go2_encode_state(next_state)
        assert history[2,4,4] == 1 and history[3,0,0] == 0
        assert features[17,0,0] > 0 and go2_encode_state(initial)[17,0,0] < 0
        checks.append('History and komi retain the evaluated current-player perspective.')
        assert go2_cache_key(initial) != go2_cache_key(passed)
        assert predictions[0]['cache_key'] != predictions[3]['cache_key']
        checks.append('Identical stones with different player/pass/history context have distinct cache references.')
        try:
            go2_eval_batch(model, [empty_terminal])
            raise AssertionError('Terminal neural evaluation was accepted.')
        except ValueError as error:
            assert 'Terminal' in str(error)
        checks.append('Terminal leaves refuse neural evaluation.')
        def fixture_policy(state, action):
            target = np.zeros(state.size*state.size+1, dtype=np.float32)
            target[action] = 1.0
            provenance = {'source':'disposable_optimizer_fixture', 'source_id':str(action),
                          'perspective':'action', 'certainty':'search_estimate',
                          'scope':'Synthetic target for gradient validation; no claim of optimality.'}
            return target, provenance
        rows = []
        for state, terminal, action in ((initial,empty_terminal,40),(passed,empty_terminal,81),(occupied,occupied_terminal,81)):
            target, provenance = fixture_policy(state, action)
            rows.append(go2_training_row(state,terminal,target,provenance,source_id='native_terminal_fixture'))
        assert rows[0]['labels']['outcome']['target'] == [1.0,0.0,0.0]
        assert rows[1]['labels']['outcome']['target'] == [0.0,0.0,1.0]
        assert rows[0]['labels']['score']['target'] < 0 < rows[1]['labels']['score']['target']
        assert np.array(rows[2]['labels']['ownership']['target']).max() == -1
        draw_state = go2_initial_state(size=3, komi=0.0)
        draw_terminal = go2_step(go2_step(draw_state, 9), 9)
        draw_row = go2_training_row(draw_state, draw_terminal, source_id='native_draw_fixture')
        assert draw_row['labels']['outcome']['target'] == [0.0,1.0,0.0]
        assert draw_row['labels']['score']['target'] == 0.0
        checks.append('Exact replay supplies loss/draw/win, score and ownership with canonical sign conversion.')
        try:
            go2_training_row(initial, occupied)
            raise AssertionError('An unfinished game supplied terminal labels.')
        except ValueError as error:
            assert 'cutoff' in str(error)
        bad_row = json.loads(json.dumps(rows[2]))
        bad_row['labels']['policy']['target'] = [0.0]*82
        bad_row['labels']['policy']['target'][40] = 1.0
        try:
            go2_joint_loss(model, [bad_row])
            raise AssertionError('An illegal target was accepted.')
        except ValueError as error:
            assert 'illegal' in str(error)
        checks.append('Cutoffs and illegal supervised targets are rejected.')
        identity_before = go2_model_identity(model)
        assert identity_before == go2_model_identity(model)
        restored_model = Go2Network(model.config)
        restored_model.load_state_dict(model.state_dict())
        assert go2_model_identity(restored_model) == identity_before
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=0.0)
        initial_loss = float(go2_joint_loss(model, rows)['loss'].detach())
        gradient_norms = {}
        for update in range(3):
            optimizer.zero_grad(set_to_none=True)
            result = go2_joint_loss(model, rows)
            result['loss'].backward()
            if update == 0:
                for name, parameters in (
                    ('encoder', model.stem.parameters()), ('scout', model.scout_board.parameters()),
                    ('worker', model.worker.parameters()), ('score', model.score_head.parameters()),
                    ('ownership', model.ownership_head.parameters()),
                ):
                    gradient_norms[name] = sum(float(parameter.grad.abs().sum()) for parameter in parameters if parameter.grad is not None)
                assert all(math.isfinite(value) and value > 0 for value in gradient_norms.values())
            optimizer.step()
        final_loss = float(go2_joint_loss(model, rows)['loss'].detach())
        assert final_loss < initial_loss
        assert go2_model_identity(model) != identity_before
        assert result['counts'] == {'policy':3,'outcome':3,'score':3,'ownership':3}
        checks.append('Every head and shared encoder receives finite nonzero gradients; three steps reduce the joint loss.')
        checks.append('Network identity is preserved by exact weight restore, stable while frozen and changes after optimizer updates.')
        # A controlled head verifies that low loss energy means a negative value,
        # rather than being incorrectly treated as a desirable low-energy state.
        with torch.no_grad():
            for parameter in model.worker.parameters(): parameter.zero_()
            model.worker[-1].bias.copy_(torch.tensor([0.0,2.0,4.0]))
        loss_favored = go2_eval_batch(model, [initial])[0]['value']
        with torch.no_grad(): model.worker[-1].bias.copy_(torch.tensor([4.0,2.0,0.0]))
        win_favored = go2_eval_batch(model, [initial])[0]['value']
        with torch.no_grad(): model.worker[-1].bias.copy_(torch.tensor([4.0,0.0,4.0]))
        draw_favored = go2_eval_batch(model, [initial])[0]
        assert loss_favored < -0.8 and win_favored > 0.8
        assert abs(draw_favored['value']) < 1e-7 and draw_favored['outcome_probabilities'][1] > 0.96
        checks.append('Loss/win/draw energy preference yields negative/positive/zero value.')
        receipt = {'schema':'ahr-go-v2-fresh-network-validation','status':'PASS',
                   'checks':checks,'check_count':len(checks),'device':'cpu',
                   'fresh_weights':True,'prior_checkpoint_loaded':False,'durable_training':False,
                   'disposable_optimizer_steps':3,'feature_count':GO2_FEATURE_COUNT,
                   'native_sizes_checked':[9,5], 'parameter_count':sum(p.numel() for p in model.parameters()),
                   'initial_fixture_loss':initial_loss,'final_fixture_loss':final_loss,
                   'gradient_l1_norms':gradient_norms,'loss_favored_value':loss_favored,'win_favored_value':win_favored,
                   'draw_favored_value':draw_favored['value'],
                   'elapsed_seconds':time.perf_counter()-start_time,
                   'source_sha256':go2_code_sha(project/'experiments/ahr_v2/02_scout_worker_network.ipynb'),
                   'sources':go2_source_identity(project)}
        go2_atomic_json(project/'reports/ahr_v2/network_validation.json',receipt)
        return receipt
    finally:
        torch.set_num_threads(prior_threads)

network_validation = go2_validate_network(PROJECT)
from IPython.display import display, HTML
summary_rows = ''.join(f'<tr><td>{name}</td><td>{value}</td></tr>' for name,value in (
    ('Status',network_validation['status']),('Checks',network_validation['check_count']),
    ('Disposable optimizer steps',network_validation['disposable_optimizer_steps']),
    ('Initial fixture loss',f"{network_validation['initial_fixture_loss']:.4f}"),
    ('Final fixture loss',f"{network_validation['final_fixture_loss']:.4f}"),
    ('Runtime (seconds)',f"{network_validation['elapsed_seconds']:.2f}"),
))
display(HTML('<table><thead><tr><th>CPU validation</th><th>Result</th></tr></thead><tbody>'+summary_rows+'</tbody></table>'))
for item in network_validation['checks']:
    print('PASS ·',item)


## Optional GPU 1 throughput check

Set `GO2_NETWORK_BENCHMARK=1` before Run All to run a roughly two-second measurement on GPU 1 using fresh default-size weights. It measures native forward computation, the complete evaluation API and joint-loss/backward computation separately. There are no optimizer updates or durable model writes. These are throughput measurements, not a training-time or strength forecast.

In [ ]:
import os

if os.environ.get('GO2_NETWORK_BENCHMARK') == '1':
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError('The requested GPU 1 benchmark requires cuda:1.')
    benchmark_device = torch.device('cuda:1')
    benchmark_rng = go2_capture_rng(cuda_devices=(1,))
    benchmark_prior_threads = torch.get_num_threads()
    torch.set_num_threads(min(2, benchmark_prior_threads))
    try:
        benchmark_model = Go2Network(Go2NetConfig()).to(benchmark_device)
        benchmark_state = go2_initial_state(size=9, komi=7.5)
        benchmark_terminal = go2_step(go2_step(benchmark_state, 81), 81)
        benchmark_states = [benchmark_state] * 32
        uniform_policy = np.asarray(go2_legal_mask(benchmark_state), dtype=np.float32)
        uniform_policy /= uniform_policy.sum()
        benchmark_provenance = {'source':'synthetic_throughput_fixture','source_id':'empty-board',
                                'perspective':'action','certainty':'search_estimate',
                                'scope':'Throughput only; no move-quality claim.'}
        benchmark_row = go2_training_row(benchmark_state, benchmark_terminal,
                                        uniform_policy, benchmark_provenance,
                                        source_id='synthetic_throughput_fixture')
        benchmark_rows = [benchmark_row] * 32
        benchmark_features = torch.as_tensor(np.stack([go2_encode_state(s) for s in benchmark_states]), device=benchmark_device)
        benchmark_model.eval()
        with torch.inference_mode():
            benchmark_model(benchmark_features)
        torch.cuda.synchronize(benchmark_device)
        benchmark_identity = go2_model_identity(benchmark_model)
        benchmark_results = {}
        benchmark_operations = (
            ('native_forward', 1.0), ('complete_evaluation_api', 0.5), ('joint_loss_backward', 0.5),
        )
        for operation_name, allotted_seconds in benchmark_operations:
            batches = 0
            started = time.perf_counter()
            while time.perf_counter()-started < allotted_seconds or batches == 0:
                if operation_name == 'native_forward':
                    with torch.inference_mode():
                        benchmark_model(benchmark_features)
                elif operation_name == 'complete_evaluation_api':
                    go2_eval_batch(benchmark_model, benchmark_states)
                else:
                    benchmark_model.zero_grad(set_to_none=True)
                    benchmark_loss = go2_joint_loss(benchmark_model, benchmark_rows)
                    benchmark_loss['loss'].backward()
                torch.cuda.synchronize(benchmark_device)
                batches += 1
            elapsed = time.perf_counter()-started
            benchmark_results[operation_name] = {'batches':batches,'positions':32*batches,
                                                 'elapsed_seconds':elapsed,'positions_per_second':32*batches/elapsed}
        assert go2_model_identity(benchmark_model) == benchmark_identity
        benchmark_receipt = {
            'schema':'ahr-go-v2-fresh-network-throughput', 'status':'PASS',
            'device':'cuda:1','gpu_name':torch.cuda.get_device_name(1),'batch_size':32,
            'config':asdict(benchmark_model.config),'parameter_count':sum(p.numel() for p in benchmark_model.parameters()),
            'fresh_weights':True,'optimizer_steps':0,'durable_training':False,
            'source_sha256':go2_code_sha(PROJECT/'experiments/ahr_v2/02_scout_worker_network.ipynb'),
            'sources':go2_source_identity(PROJECT),
            'measurements':benchmark_results,
            'scope':'Short native-shape throughput only. No game-strength or long-run estimate.',
        }
        go2_atomic_json(PROJECT/'reports/ahr_v2/network_throughput.json',benchmark_receipt)
        throughput_rows = ''.join('<tr><td>'+name+'</td><td>'+f"{result['positions_per_second']:.0f}"+'</td><td>'+f"{result['elapsed_seconds']:.2f}"+'</td></tr>' for name,result in benchmark_results.items())
        display(HTML('<table><thead><tr><th>GPU 1 measurement</th><th>Positions / second</th><th>Seconds</th></tr></thead><tbody>'+throughput_rows+'</tbody></table>'))
    finally:
        if 'benchmark_model' in globals():
            del benchmark_model
        if 'benchmark_features' in globals():
            del benchmark_features
        if 'benchmark_loss' in globals():
            del benchmark_loss
        with torch.cuda.device(benchmark_device):
            torch.cuda.empty_cache()
        go2_restore_rng(benchmark_rng)
        torch.set_num_threads(benchmark_prior_threads)
else:
    print('Optional GPU 1 benchmark is off; normal Run All remains CPU only.')


## What this milestone establishes

The saved checks establish neural/search interface correctness on small native fixtures, exact legal masking and current-player label conventions. They do not establish tactical competence or game strength. The next notebook combines these fresh energy heads with adversarial tree search. Training data, teacher selection and any long GPU run require their own measured learning and recovery checks.